In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import torch

from energia import config
from energia.dataset import cargar_datos_modelo
from energia.entrenamiento import dispositivo, predecir
from energia.metricas import error_por_hora, naive_estacional
from energia.modelos import tcn
from energia.optimizacion import experimento, etapa, mejor, historial, cargar_modelo
from energia.visualizacion import estilo, guardar_figura, curvas_aprendizaje, pronostico_vs_real

estilo()
RED = "tcn"
print("torch", torch.__version__, "| dispositivo:", dispositivo())
datos = cargar_datos_modelo()

In [ ]:
MODELO_FASE2 = {"canales": 32, "kernel": 3, "niveles": 6, "dropout": 0.1, "cabeza": 64}
ENTRENAMIENTO_FASE2 = {"optimizador": "adam", "lr": 1e-3, "epocas": 50, "batch": 256, "paciencia": 8, "clip": 1.0}

In [ ]:
LR_INICIAL = {"sgd": 1e-2, "momentum": 1e-2, "nesterov": 1e-2, "adagrad": 1e-2,
              "rmsprop": 1e-3, "adam": 1e-3, "nadam": 1e-3, "adamax": 2e-3}

for opt, lr in LR_INICIAL.items():
    experimento(RED, "1_optimizador", opt, tcn, datos, MODELO_FASE2,
                {**ENTRENAMIENTO_FASE2, "optimizador": opt, "lr": lr})

etapa(RED, "1_optimizador")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for opt in LR_INICIAL:
    h = historial(RED, "1_optimizador", opt)
    axes[0].plot(h.index, h["train"], marker=".", label=opt)
    axes[1].plot(h.index, h["val"], marker=".", label=opt)
axes[0].set(title="Entrenamiento", xlabel="época", ylabel="MSE (escala estandarizada)", yscale="log")
axes[1].set(title="Validación", xlabel="época", yscale="log")
axes[1].legend(ncol=2)
fig.suptitle("TCN: pérdida por época según el optimizador")
fig.tight_layout()
guardar_figura(fig, "f3_01_tcn_optimizadores")
plt.show()

In [ ]:
E1 = mejor(RED, "1_optimizador")
OPT, LR = E1["nombre"], E1["hp_entrenamiento"]["lr"]
print("optimizador:", OPT, "| lr:", LR)

for lr in [LR, LR / 3]:
    for dropout in [0.1, 0.3]:
        for wd in [0.0, 1e-4]:
            experimento(RED, "2_regularizacion", f"lr{lr:.0e}_do{dropout}_wd{wd:.0e}", tcn, datos,
                        {**MODELO_FASE2, "dropout": dropout},
                        {**ENTRENAMIENTO_FASE2, "optimizador": OPT, "lr": lr, "weight_decay": wd})

etapa(RED, "2_regularizacion")

In [ ]:
E2 = mejor(RED, "2_regularizacion")
M2, T2 = E2["hp_modelo"], E2["hp_entrenamiento"]
print(E2["nombre"], M2, T2)

NAIVE = {**M2, "corregir_naive": True}
VARIANTES = {
    "canales64": ({**M2, "canales": 64}, T2),
    "cabeza128": ({**M2, "cabeza": 128}, T2),
    "corregir_naive": (NAIVE, T2),
    "corregir_naive_adam": (NAIVE, {**T2, "optimizador": "adam", "lr": 1e-3}),
    "corregir_naive_150": (NAIVE, {**T2, "epocas": 150}),
}
for nombre, (hp_m, hp_t) in VARIANTES.items():
    experimento(RED, "3_arquitectura", nombre, tcn, datos, hp_m, hp_t)

pd.concat([etapa(RED, "2_regularizacion").head(1).assign(etapa="2_regularizacion"),
           etapa(RED, "3_arquitectura").assign(etapa="3_arquitectura")], ignore_index=True)

In [ ]:
FINAL = mejor(RED, ["2_regularizacion", "3_arquitectura"])
MF, TF = FINAL["hp_modelo"], FINAL["hp_entrenamiento"]
NOMBRE = FINAL["nombre"]
print(FINAL["etapa"], NOMBRE)
print(MF)
print(TF)

SEMILLAS = [42, 7, 2026]
for s in SEMILLAS:
    experimento(RED, "4_final", NOMBRE, tcn, datos, MF, TF, semilla=s, prueba=True, guardar=True)

final = etapa(RED, "4_final").query("nombre == @NOMBRE")
final

In [ ]:
fase2 = pd.read_csv(config.RESULTADOS / "fase2" / RED / "metricas.csv", index_col=[0, 1])
filas = {}
for split in ["val", "test"]:
    for c in ["demanda_MAPE_%", "precio_MAPE_%", "precio_RMSE", "precio_R2"]:
        filas[(split, c)] = {
            "TCN fase 2": fase2.loc[(RED, split), c],
            "TCN fase 3 (media)": final[f"{split}_{c}"].mean(),
            "TCN fase 3 (desv.)": final[f"{split}_{c}"].std(),
            "naive m=24": fase2.loc[("naive_m24", split), c],
            "naive m=168": fase2.loc[("naive_m168", split), c],
        }
comparacion = pd.DataFrame(filas).T.round(3)
comparacion.to_csv(config.RESULTADOS / "fase3" / RED / "comparacion_final.csv")
comparacion

In [ ]:
fig = curvas_aprendizaje(historial(RED, "4_final", NOMBRE, SEMILLAS[0]), "TCN optimizada: pérdida por época")
guardar_figura(fig, "f3_02_tcn_curvas")
plt.show()

In [ ]:
modelo = cargar_modelo(RED, "4_final", NOMBRE, tcn, datos, semilla=SEMILLAS[0])
predicciones = {s: predecir(modelo, datos, s) for s in ["val", "test"]}

fig = pronostico_vs_real(datos, predicciones["val"], "val", dias=14, titulo="TCN optimizada: pronóstico vs real (validación, 14 días)")
guardar_figura(fig, "f3_03_tcn_pronostico_val")
plt.show()

fig = pronostico_vs_real(datos, predicciones["test"], "test", dias=14, titulo="TCN optimizada: pronóstico vs real (prueba, 14 días)")
guardar_figura(fig, "f3_04_tcn_pronostico_test")
plt.show()

In [ ]:
paquete = torch.load(config.MODELOS / "tcn.pt", map_location="cpu")
modelo_f2 = tcn.construir(datos, **{k: v for k, v in paquete["hiperparametros"].items() if k in tcn.HIPERPARAMETROS})
modelo_f2.load_state_dict(paquete["estado"])

y_val = datos.y_original("val")
errores = {
    "TCN fase 3": error_por_hora(y_val, predicciones["val"]),
    "TCN fase 2": error_por_hora(y_val, predecir(modelo_f2, datos, "val")),
    "naive_m24": error_por_hora(y_val, naive_estacional(datos, "val", m=24)),
    "naive_m168": error_por_hora(y_val, naive_estacional(datos, "val", m=168)),
}
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
for nombre, e in errores.items():
    axes[0].plot(e.index - 1, e["demanda"], marker=".", label=nombre)
    axes[1].plot(e.index - 1, e["precio"], marker=".", label=nombre)
axes[0].set(title="MAE demanda por hora (validación)", xlabel="hora del día", ylabel="kWh")
axes[1].set(title="MAE precio por hora (validación)", xlabel="hora del día", ylabel="COP/kWh")
axes[1].legend()
fig.tight_layout()
guardar_figura(fig, "f3_05_tcn_error_por_hora")
plt.show()